# Why reinforcement learning can improve long sequence generation

This notebook builds a tiny, fully inspectable language-modeling problem.  The alphabet is deliberately small and the desired answer is a long string.  The key lesson is that **choosing the most likely next token at every step (greedy decoding) is not the same as choosing the sequence with the best long-run outcome**.

We will:

1. Define a reference language model that knows the correct token at each position but gives a slightly higher next-token probability to a tempting distractor.
2. Show that greedy decoding from this reference model fails badly.
3. Treat sequence generation as a reinforcement-learning problem with a terminal reward for matching the long target.
4. Train a small policy with REINFORCE and show that its greedy output beats the reference model's greedy output.

This is a toy example, but it mirrors the reason RL-style tuning can help real LLMs: the training signal can optimize a whole response rather than only one locally likely token at a time.


In [1]:
import math
import random
from collections import Counter

random.seed(7)

ALPHABET = list("ABCD")
TARGET = "ABACDABCDABACDABCDABACDA"  # long ground-truth sequence, length 25
L = len(TARGET)

print("Alphabet:", ALPHABET)
print("Target length:", L)
print("Target:", TARGET)


Alphabet: ['A', 'B', 'C', 'D']
Target length: 24
Target: ABACDABCDABACDABCDABACDA


## A reference next-token model with a local trap

At each correct prefix, the reference model assigns:

- probability `0.49` to the target token, and
- probability `0.51` to a distractor token.

So the reference model contains useful information: the correct token is almost as likely as the top token.  However, greedy decoding always takes the `0.51` option.  That locally optimal choice can be globally poor when the reward depends on the full sequence.


In [2]:
def distractor_for(token):
    # A deterministic wrong token in the same tiny alphabet.
    return ALPHABET[(ALPHABET.index(token) + 1) % len(ALPHABET)]

REFERENCE_PROBS = []
for target_token in TARGET:
    probs = {ch: 0.0 for ch in ALPHABET}
    probs[target_token] = 0.49
    probs[distractor_for(target_token)] = 0.51
    REFERENCE_PROBS.append(probs)

def greedy_from_probs(probs_by_position):
    return "".join(max(probs, key=probs.get) for probs in probs_by_position)

def match_fraction(sequence, target=TARGET):
    return sum(a == b for a, b in zip(sequence, target)) / len(target)

reference_greedy = greedy_from_probs(REFERENCE_PROBS)
print("Reference greedy output:", reference_greedy)
print("Target:                 ", TARGET)
print("Reference greedy reward:", match_fraction(reference_greedy))
print("Reference target-token probabilities, first 8 positions:")
for i in range(8):
    print(i, TARGET[i], REFERENCE_PROBS[i])


Reference greedy output: BCBDABCDABCBDABCDABCBDAB
Target:                  ABACDABCDABACDABCDABACDA
Reference greedy reward: 0.0
Reference target-token probabilities, first 8 positions:
0 A {'A': 0.49, 'B': 0.51, 'C': 0.0, 'D': 0.0}
1 B {'A': 0.0, 'B': 0.49, 'C': 0.51, 'D': 0.0}
2 A {'A': 0.49, 'B': 0.51, 'C': 0.0, 'D': 0.0}
3 C {'A': 0.0, 'B': 0.0, 'C': 0.49, 'D': 0.51}
4 D {'A': 0.51, 'B': 0.0, 'C': 0.0, 'D': 0.49}
5 A {'A': 0.49, 'B': 0.51, 'C': 0.0, 'D': 0.0}
6 B {'A': 0.0, 'B': 0.49, 'C': 0.51, 'D': 0.0}
7 C {'A': 0.0, 'B': 0.0, 'C': 0.49, 'D': 0.51}


## Why sampling can reveal the better long-run behavior

Although the target token is never the greedy token, it is sampled often because its probability is `0.49`.  A reinforcement learner can use the reward from complete sampled sequences to increase the probability of tokens that tend to appear in high-reward sequences.

The reward below is the fraction of positions that match the target.  This dense toy reward keeps the notebook short and stable; in real alignment work, rewards can come from human preferences, tests, verifiers, or learned reward models.


In [3]:
def sample_from_probs(probs_by_position):
    output = []
    for probs in probs_by_position:
        r = random.random()
        total = 0.0
        for ch, p in probs.items():
            total += p
            if r <= total:
                output.append(ch)
                break
    return "".join(output)

samples = [sample_from_probs(REFERENCE_PROBS) for _ in range(1000)]
rewards = [match_fraction(s) for s in samples]
print("Mean reward from reference sampling:", round(sum(rewards) / len(rewards), 3))
print("Best reward among 1,000 samples:", round(max(rewards), 3))
print("One high-reward sample:", samples[rewards.index(max(rewards))])
print("Reference greedy reward:", match_fraction(reference_greedy))


Mean reward from reference sampling: 0.492
Best reward among 1,000 samples: 0.792
One high-reward sample: BBACDABDDBBBCDABCDABACAA
Reference greedy reward: 0.0


## Reinforcement-learning setup

We now train a tabular policy: one categorical distribution over the alphabet for each position.  It starts from the reference model's probabilities.  Then we use the REINFORCE policy-gradient estimator:

\[

abla \log \pi(a_t \mid t) (R - b)
\]

where `R` is the sequence-level reward and `b` is a moving-average baseline.  The policy receives no supervised label during the update; it only receives the reward for the completed sequence.


In [4]:
def logit(p):
    return math.log(p)

# Start from the reference probabilities.
logits = []
for probs in REFERENCE_PROBS:
    logits.append([logit(max(probs[ch], 1e-12)) for ch in ALPHABET])

def softmax(row):
    m = max(row)
    exps = [math.exp(x - m) for x in row]
    z = sum(exps)
    return [x / z for x in exps]

def policy_probs():
    return [dict(zip(ALPHABET, softmax(row))) for row in logits]

def sample_policy():
    probs_by_pos = policy_probs()
    output, chosen_indices = [], []
    for probs in probs_by_pos:
        r = random.random()
        total = 0.0
        for j, ch in enumerate(ALPHABET):
            total += probs[ch]
            if r <= total:
                output.append(ch)
                chosen_indices.append(j)
                break
    return "".join(output), chosen_indices, probs_by_pos

def greedy_policy():
    return greedy_from_probs(policy_probs())

learning_rate = 0.18
baseline = 0.0
history = []

for step in range(1, 5001):
    sequence, chosen_indices, probs_by_pos = sample_policy()
    reward = match_fraction(sequence)
    baseline = 0.98 * baseline + 0.02 * reward
    advantage = reward - baseline

    # Gradient of log softmax for the sampled action.
    for t, action_index in enumerate(chosen_indices):
        probs = [probs_by_pos[t][ch] for ch in ALPHABET]
        for j in range(len(ALPHABET)):
            grad = (1.0 if j == action_index else 0.0) - probs[j]
            logits[t][j] += learning_rate * advantage * grad

    if step % 250 == 0:
        greedy = greedy_policy()
        history.append((step, reward, baseline, match_fraction(greedy), greedy))

print("step | sampled reward | baseline | greedy reward | greedy output")
for row in history[::4] + history[-4:]:
    step, reward, base, greedy_reward, greedy = row
    print(f"{step:4d} | {reward:14.3f} | {base:8.3f} | {greedy_reward:13.3f} | {greedy}")


step | sampled reward | baseline | greedy reward | greedy output
 250 |          0.750 |    0.608 |         1.000 | ABACDABCDABACDABCDABACDA
1250 |          0.958 |    0.925 |         1.000 | ABACDABCDABACDABCDABACDA
2250 |          0.958 |    0.959 |         1.000 | ABACDABCDABACDABCDABACDA
3250 |          1.000 |    0.973 |         1.000 | ABACDABCDABACDABCDABACDA
4250 |          0.917 |    0.981 |         1.000 | ABACDABCDABACDABCDABACDA
4250 |          0.917 |    0.981 |         1.000 | ABACDABCDABACDABCDABACDA
4500 |          1.000 |    0.987 |         1.000 | ABACDABCDABACDABCDABACDA
4750 |          0.958 |    0.983 |         1.000 | ABACDABCDABACDABCDABACDA
5000 |          1.000 |    0.984 |         1.000 | ABACDABCDABACDABCDABACDA


## Final comparison

The reference model's greedy decoder follows the best next token and gets the entire sequence wrong.  The RL-tuned policy has learned that the slightly less likely local token gives a better full-sequence reward, so its greedy decoder switches to the target string.


In [5]:
final_probs = policy_probs()
rl_greedy = greedy_policy()

print("Target:                  ", TARGET)
print("Reference greedy output: ", reference_greedy)
print("RL-tuned greedy output:  ", rl_greedy)
print()
print("Reference greedy reward:", match_fraction(reference_greedy))
print("RL-tuned greedy reward: ", match_fraction(rl_greedy))
print()
print("First 8 target-token probabilities before and after RL:")
for i in range(8):
    token = TARGET[i]
    before = REFERENCE_PROBS[i][token]
    after = final_probs[i][token]
    print(f"position {i:2d}, token {token}: before={before:.3f}, after={after:.3f}")


Target:                   ABACDABCDABACDABCDABACDA
Reference greedy output:  BCBDABCDABCBDABCDABCBDAB
RL-tuned greedy output:   ABACDABCDABACDABCDABACDA

Reference greedy reward: 0.0
RL-tuned greedy reward:  1.0

First 8 target-token probabilities before and after RL:
position  0, token A: before=0.490, after=0.987
position  1, token B: before=0.490, after=0.986
position  2, token A: before=0.490, after=0.984
position  3, token C: before=0.490, after=0.985
position  4, token D: before=0.490, after=0.987
position  5, token A: before=0.490, after=0.983
position  6, token B: before=0.490, after=0.988
position  7, token C: before=0.490, after=0.982


## Takeaways

- Greedy decoding optimizes a local criterion: the next token with highest probability.
- Long sequence quality is a global criterion: the whole completion must be good.
- A model can contain enough signal for a good answer while greedy decoding still extracts a poor answer.
- Reinforcement learning can move probability mass toward tokens that produce better complete sequences, even when those tokens are not initially the top next-token choices.

This notebook intentionally avoids large neural networks so the mechanism is visible.  Real LLM RL fine-tuning uses richer policies, prompts, reward models, KL penalties, and batching, but the core idea is the same: optimize behavior using feedback on complete generations.
